In [1]:
import pandas as pd
df = pd.read_csv('final_preprocessed_data.csv')

In [2]:
df = df.drop(columns=[
'SEASON_ID_home', 'TEAM_ID_home', 'TEAM_NAME_home', 'GAME_ID','GAME_DATE', 'TEAM_ID_away','TEAM_NAME_away','HOME_ELO_POST','AWAY_ELO_POST','WL'])
df = df.dropna()

In [6]:
# ...existing code...
target = "TARGET_COLUMN"  # Replace with your actual target column name
correlations = df.corr()['WIN'].sort_values(ascending=False)
correlations

WIN                             1.000000
HOME_WIN_PROB                   0.358438
HOME_ELO_PRE                    0.282526
PLUS_MINUS_cumulative_home      0.268540
NET_RATING_cumulative_home      0.267085
                                  ...   
missing_players_home           -0.190047
NET_RATING_cumulative_away     -0.217411
E_NET_RATING_cumulative_away   -0.218879
PLUS_MINUS_cumulative_away     -0.219052
AWAY_ELO_PRE                   -0.236785
Name: WIN, Length: 170, dtype: float64

# Split and scale data

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split the data into features and target
X = df.drop(columns=['WIN'])
y = df['WIN']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# LASSO

In [12]:
from sklearn.linear_model import LassoCV

lasso = LassoCV(alphas=[0.1, 0.01, 0.001], cv=5, random_state=42, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

# Print the coefficients
coefficients = pd.Series(lasso.coef_, index=X.columns)
coefficients[coefficients != 0].sort_values(ascending=False)

HOME_WIN_PROB                 0.119201
DREB_PCT_cumulative_home      0.022167
OFF_RATING_rolling_home       0.018525
AST_rolling_away              0.013236
missing_impact_away           0.011497
PLUS_MINUS_cumulative_home    0.007612
REB_RATIO_cumulative_away     0.005591
missing_star_away             0.005114
AST_PCT_cumulative_away       0.003437
AST_TO_RATIO_rolling_away     0.001735
AST_TOV_rolling_away          0.000425
TS_PCT_cumulative_home        0.000003
TM_TOV_PCT_rolling_away      -0.000804
FT_PCT_rolling_away          -0.002780
OREB_cumulative_home         -0.004412
PF_rolling_home              -0.006100
FT_PCT_rolling_home          -0.008074
DEF_RATING_rolling_away      -0.008089
FG3_PCT_cumulative_away      -0.008899
TS_PCT_cumulative_away       -0.009090
PACE_rolling_home            -0.011633
FG3A_rolling_away            -0.013711
TM_TOV_PCT_cumulative_home   -0.015545
AWAY_ELO_PRE                 -0.015733
REB_RATIO_rolling_home       -0.020183
STL_rolling_away         

# PCA

In [14]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Number of PCA components to explain 95% variance: {pca.n_components_}")

explained_var = pd.Series(pca.explained_variance_ratio_)
print("Explained variance ratio per component:")
print(explained_var)

loadings = pd.DataFrame(pca.components_.T, index=X_train.columns, columns=[f'PC{i+1}' for i in range(pca.n_components_)])

loadings[['PC1', 'PC2']].reindex(loadings['PC1'].abs().sort_values(ascending=False).index)

Number of PCA components to explain 95% variance: 32
Explained variance ratio per component:
0     0.302539
1     0.102705
2     0.094931
3     0.046137
4     0.045269
5     0.032602
6     0.030813
7     0.029207
8     0.026298
9     0.021473
10    0.020344
11    0.019256
12    0.018194
13    0.015087
14    0.013766
15    0.012534
16    0.011958
17    0.011417
18    0.010507
19    0.009347
20    0.008956
21    0.008102
22    0.007851
23    0.007518
24    0.007185
25    0.006744
26    0.006347
27    0.005535
28    0.005193
29    0.004912
30    0.004832
31    0.004335
dtype: float64


,PC1,PC2
E_OFF_RATING_cumulative_away,-0.139278,-0.003702
OFF_RATING_cumulative_away,-0.139273,-0.004201
PTS_cumulative_away,-0.139159,-0.003848
OFF_RATING_cumulative_home,-0.139151,-0.015544
E_OFF_RATING_cumulative_home,-0.139147,-0.015548
...,...,...
BLK_rolling_away,0.000806,0.060101
REB_RATIO_cumulative_home,0.000769,0.008123
FGM_rolling_away,-0.000416,0.179553
FG3A_rolling_away,0.000369,0.029497
